# Notebook 02 — Seller-Level SCS (Distribution vs Confidence)

Study A. Compute seller-level Seg assignment via confidence-weighted voting
(CWV) and quantify assignment confidence with two families of SCS, then
decompose into within-store and between-store components.

Two SCS families are computed and contrasted (this contrast is an RQ1 result):
- Distribution-based (first paper): n_dom / N * 1/(1+H), confidence NOT used.
- Confidence-based (Study A extension): mean confidence * 1/(1+H).

Target of validation: seller-level ASSIGNMENT accuracy (does CWV recover the
seller's ground-truth dominant segment?), following the first paper's Table 9,
not item-level accuracy. A single-store seller has between-store term = 1,
recovering the first-paper store-level case.

Metric variants:
- Pooled baseline (treat seller as one store)
- Decomposed main: SCS_s = W_s * B_s
- computed under both the distribution and confidence within-formulations.


In [1]:
# %% ============================================================
# Notebook 02 — Seller-Level SCS (Distribution vs Confidence)
# Imports, paths, load simulation outputs
# ============================================================
import os
import json
import numpy as np
import pandas as pd

SEED = 42
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
ARTIFACT_DIR = os.path.join(ROOT, "artifacts")
TAB_DIR = os.path.join(ROOT, "results", "tables")

seller_items = pd.read_csv(os.path.join(ARTIFACT_DIR, "seller_items.csv"))
seller_meta = pd.read_csv(os.path.join(ARTIFACT_DIR, "seller_meta.csv"))
proba = np.load(os.path.join(ARTIFACT_DIR, "test_proba.npy"))
N_CLASSES = proba.shape[1]
print("seller-item rows:", seller_items.shape[0], "| sellers:", seller_meta.shape[0])
gt_map = dict(zip(seller_meta["seller_id"], seller_meta["gt_segment"]))

# %% ============================================================
# Shared entropy helpers on a categorical distribution.
# ============================================================
def norm_entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    if p.size <= 1:
        return 0.0
    h = -(p * np.log(p)).sum()
    return float(h / np.log(len(p)))


def raw_entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    if p.size == 0:
        return 0.0
    return float(-(p * np.log(p)).sum())


def cat_counts(labels, n_classes):
    return np.bincount(labels, minlength=n_classes).astype(float)

# %% ============================================================
# CWV assignment: the seller's assigned segment is the confidence-
# weighted plurality of item-level predictions. Assignment is correct
# if it matches the seller's ground-truth dominant category.
# ============================================================
def cwv_assign(g, n_classes):
    weighted = np.zeros(n_classes)
    for c, w in zip(g["pred"].to_numpy(), g["confidence"].to_numpy()):
        weighted[c] += w
    return int(np.argmax(weighted))

# %% ============================================================
# SCS building blocks.
# Distribution within: n_dom/N * 1/(1+H_pred)   (first paper form)
# Confidence within  : mean_conf * 1/(1+H_pred) (Study A form)
# Between term B_s    : 1/(1+H_between) over per-store dominant preds.
# Single store -> H_between = 0 -> B_s = 1 (first-paper reduction).
# ============================================================
def within_terms(g, n_classes):
    pred = g["pred"].to_numpy()
    conf = g["confidence"].to_numpy()
    counts = cat_counts(pred, n_classes)
    N = counts.sum()
    n_dom = counts.max()
    H = raw_entropy(counts / N) if N > 0 else 0.0
    dist_within = (n_dom / N) * (1.0 / (1.0 + H)) if N > 0 else 0.0
    conf_within = conf.mean() * (1.0 / (1.0 + H)) if N > 0 else 0.0
    return dist_within, conf_within


def between_term(g, n_classes):
    store_dom = []
    for _, s in g.groupby("store_id"):
        c = cat_counts(s["pred"].to_numpy(), n_classes)
        store_dom.append(int(np.argmax(c)))
    dom_dist = cat_counts(np.array(store_dom), n_classes)
    dom_dist = dom_dist / dom_dist.sum()
    H_btw = raw_entropy(dom_dist)
    return 1.0 / (1.0 + H_btw), H_btw

# %% ============================================================
# Compute per-seller: CWV assignment, assignment correctness, and all
# SCS variants (pooled and decomposed, distribution and confidence).
# ============================================================
records = []
for sid, g in seller_items.groupby("seller_id"):
    gt = gt_map[sid]
    assigned = cwv_assign(g, N_CLASSES)
    assign_correct = int(assigned == gt)

    # Pooled (whole seller as one store)
    dist_pool, conf_pool = within_terms(g, N_CLASSES)

    # Decomposed: item-weighted within across stores * between term
    store_dist, store_conf, store_w = [], [], []
    for _, s in g.groupby("store_id"):
        dw, cw = within_terms(s, N_CLASSES)
        store_dist.append(dw); store_conf.append(cw); store_w.append(len(s))
    store_w = np.array(store_w, dtype=float)
    W_dist = float(np.average(store_dist, weights=store_w))
    W_conf = float(np.average(store_conf, weights=store_w))
    B_s, H_btw = between_term(g, N_CLASSES)

    records.append({
        "seller_id": sid,
        "assigned": assigned,
        "assign_correct": assign_correct,
        "scs_dist_pooled": dist_pool,
        "scs_conf_pooled": conf_pool,
        "scs_dist_decomp": W_dist * B_s,
        "scs_conf_decomp": W_conf * B_s,
        "W_dist": W_dist,
        "W_conf": W_conf,
        "B_s": B_s,
        "H_between": H_btw,
        "item_accuracy": g["correct"].mean(),
    })

scs_df = pd.DataFrame(records).merge(seller_meta, on="seller_id", how="left")
scs_df.to_csv(os.path.join(ARTIFACT_DIR, "seller_scs.csv"), index=False)
print("SCS computed for", scs_df.shape[0], "sellers")
print("Overall CWV assignment accuracy:", round(scs_df["assign_correct"].mean(), 4))
print(scs_df[["scs_dist_pooled","scs_conf_pooled","scs_dist_decomp",
              "scs_conf_decomp","assign_correct"]].describe().round(4))

# %% ============================================================
# Reducibility: single-store sellers must have B_s = 1.
# ============================================================
single = scs_df[scs_df["n_stores"] == 1]
print("Single-store sellers:", len(single))
print("All B_s == 1 for single store:", bool(np.allclose(single["B_s"], 1.0)))

# %% ============================================================
# RQ1 core contrast: which SCS family better tracks seller-level
# ASSIGNMENT reliability? Correlate each SCS variant with assignment
# correctness (point-biserial via Spearman) on Method A (natural).
# ============================================================
from scipy.stats import spearmanr, pointbiserialr

a = scs_df[scs_df["method"] == "A_real"]
variants = ["scs_dist_pooled", "scs_conf_pooled", "scs_dist_decomp", "scs_conf_decomp"]
rows = []
for v in variants:
    sp, _ = spearmanr(a[v], a["assign_correct"])
    pb, _ = pointbiserialr(a["assign_correct"], a[v])
    rows.append({"scs_variant": v, "spearman_vs_assign": sp, "pointbiserial": pb})
corr_tbl = pd.DataFrame(rows)
corr_tbl.to_csv(os.path.join(TAB_DIR, "table_scs_assignment_correlation.csv"), index=False)
print("SCS vs seller-level assignment correctness (Method A):")
print(corr_tbl.round(4).to_string(index=False))

# %% ============================================================
# Bin-level assignment accuracy (first-paper Table 9 analogue) for the
# main decomposed metric under BOTH SCS families. Bins are fixed SCS
# ranges so the two families are directly comparable.
# ============================================================
def bin_assignment_accuracy(df, metric, edges):
    lab = pd.cut(df[metric], edges, include_lowest=True)
    t = df.groupby(lab, observed=True).agg(
        n=("assign_correct", "size"),
        mean_scs=(metric, "mean"),
        assign_acc=("assign_correct", "mean"),
    ).reset_index(drop=True)
    return t

edges = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
for fam, metric in [("distribution", "scs_dist_decomp"), ("confidence", "scs_conf_decomp")]:
    t = bin_assignment_accuracy(a, metric, edges)
    t.to_csv(os.path.join(TAB_DIR, f"table_scs_monotonicity_{fam}.csv"), index=False)
    diffs = np.diff(t["assign_acc"].to_numpy())
    print(f"[{fam}] bin-level assignment accuracy:")
    print(t.round(4).to_string(index=False))
    print("  non-decreasing across bins:", bool((diffs >= -1e-9).all()))
    print()

# %% ============================================================
# Within/between diagnosis (RQ2 material): among misassigned sellers,
# is low within (store-internal mixing) or low between (cross-store
# disagreement) the dominant source of uncertainty?
# ============================================================
mis = scs_df[(scs_df["method"] == "A_real") & (scs_df["assign_correct"] == 0)]
cor = scs_df[(scs_df["method"] == "A_real") & (scs_df["assign_correct"] == 1)]
diag = pd.DataFrame({
    "group": ["misassigned", "correct"],
    "n": [len(mis), len(cor)],
    "mean_W_dist": [mis["W_dist"].mean(), cor["W_dist"].mean()],
    "mean_B_s": [mis["B_s"].mean(), cor["B_s"].mean()],
    "mean_H_between": [mis["H_between"].mean(), cor["H_between"].mean()],
})
diag.to_csv(os.path.join(TAB_DIR, "table_within_between_diagnosis.csv"), index=False)
print(diag.round(4).to_string(index=False))

seller-item rows: 412560 | sellers: 2880
SCS computed for 2880 sellers
Overall CWV assignment accuracy: 0.7174
       scs_dist_pooled  scs_conf_pooled  scs_dist_decomp  scs_conf_decomp  \
count        2880.0000        2880.0000        2880.0000        2880.0000   
mean            0.1825           0.3592           0.1915           0.3318   
std             0.1120           0.0700           0.1292           0.1254   
min             0.0423           0.2602           0.0303           0.1176   
25%             0.1003           0.3101           0.0935           0.2168   
50%             0.1463           0.3370           0.1488           0.3359   
75%             0.2480           0.3898           0.2658           0.4132   
max             0.7497           0.7756           0.7861           0.8206   

       assign_correct  
count       2880.0000  
mean           0.7174  
std            0.4504  
min            0.0000  
25%            0.0000  
50%            1.0000  
75%            1.0000  
max